# 15 — Streamlit Backend Test

**Objective**: validate `app/backend.py`'s cached data-access functions independently of the Streamlit UI runtime, then validate every page in `app/pages/` actually runs without exceptions using Streamlit's own headless testing framework (`streamlit.testing.v1.AppTest`) — which executes each page's real Python script in a simulated Streamlit runtime, not a mock.

**Dependencies**: everything through Phase 9 (`app/backend.py` is a thin, cached wrapper over the LangGraph pipeline — see its module docstring).

**On verifying a browser UI without a browser**: this environment has no supported headless-Chromium binary (`chromium-cli` isn't installed; Playwright's bundled Chromium doesn't support this sandbox's macOS version — confirmed by attempting the install, not assumed). `AppTest` is Streamlit's own first-class alternative: it runs the actual page script against a simulated `ScriptRunContext` and returns the resulting element tree, so it catches real Python exceptions, wrong values, and missing data exactly as they'd occur in a browser session, just without pixels. A real `streamlit run` server was also launched and its `/_stcore/health` endpoint checked directly (below) as an additional, real-process layer beyond the simulated harness.

In [1]:
import os
import sys
import shutil
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

shutil.rmtree(PROJECT_ROOT / "data/snapshots", ignore_errors=True)
(PROJECT_ROOT / "data/snapshots").mkdir(parents=True, exist_ok=True)

from datetime import date
from app import backend

print("Ready.")

2026-09-05 13:18:02.783 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


2026-09-05 13:18:02.786 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


Ready.


## Cached pipeline access

In [2]:
AS_OF = date(2026, 9, 15)
result = backend.run_portfolio_query(AS_OF)
print(f"unified_projects: {len(result['unified_projects'])}")
print(f"confidence: {result['confidence']}")
print(f"validation_passed: {result['validation_passed']}")

2026-09-05 13:18:02.795 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


2026-09-05 13:18:02.796 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


{"request_id": "streamlit-portfolio-2026-09-15", "timestamp": "2026-09-05T17:18:02.865868+00:00", "event": "graph_node_start", "node": "classify_request"}
{"request_id": "streamlit-portfolio-2026-09-15", "timestamp": "2026-09-05T17:18:02.866025+00:00", "event": "graph_node_end", "node": "classify_request", "latency_ms": 0.02}
{"request_id": "streamlit-portfolio-2026-09-15", "timestamp": "2026-09-05T17:18:02.867131+00:00", "event": "graph_node_start", "node": "fetch_delivery_data"}
{"request_id": "streamlit-portfolio-2026-09-15", "timestamp": "2026-09-05T17:18:02.888640+00:00", "event": "records_retrieved", "source": "Jira", "issue_count": 270, "sprint_count": 20, "partial_failure": false}
{"request_id": "streamlit-portfolio-2026-09-15", "timestamp": "2026-09-05T17:18:02.888713+00:00", "event": "graph_node_end", "node": "fetch_delivery_data", "latency_ms": 21.52}
{"request_id": "streamlit-portfolio-2026-09-15", "timestamp": "2026-09-05T17:18:02.889261+00:00", "event": "graph_node_start"

In [3]:
# Cache correctness: same date -> same result, not recomputed
import time
start = time.time()
result2 = backend.run_portfolio_query(AS_OF)
elapsed = time.time() - start
print(f"Second call for the same date took {elapsed*1000:.1f}ms (cached, not re-run)")
assert result2["confidence"] == result["confidence"]

Second call for the same date took 2.6ms (cached, not re-run)


## DataFrame conversion helpers

In [4]:
projects_df = backend.projects_dataframe(result["unified_projects"])
risks_df = backend.risks_dataframe(result["risks"])
blocked_df = backend.blocked_issues_dataframe(result["jira_issues"])

print(projects_df[["Project", "RAG", "Budget Consumption %", "Delivery Progress %"]])
print(f"\n{len(risks_df)} risk rows, {len(blocked_df)} blocked-issue rows")

                          Project    RAG  Budget Consumption %  \
0  Phoenix Platform Modernization    RED            111.919743   
1           Orca Payments Gateway    RED             80.966986   
2            Nova Customer Portal    RED            113.672012   
3          Titan Infra Automation    RED            102.361839   
4             Lynx Data Analytics    RED            106.219932   
5   Quasar Self-Service Analytics  AMBER                   NaN   
6       Helios Compliance Program  AMBER             64.554455   

   Delivery Progress %  
0            37.903226  
1            30.379747  
2            36.111111  
3            49.128920  
4            45.318352  
5            64.285714  
6                  NaN  

21 risk rows, 38 blocked-issue rows


## Filtering

In [5]:
red_only = backend.apply_project_filters(projects_df, rag=["RED"])
print(f"RED-filtered: {red_only['Project'].tolist()}")

amber_only = backend.apply_project_filters(projects_df, rag=["AMBER"])
print(f"AMBER-filtered: {amber_only['Project'].tolist()}")

assert len(red_only) + len(amber_only) <= len(projects_df)

RED-filtered: ['Phoenix Platform Modernization', 'Orca Payments Gateway', 'Nova Customer Portal', 'Titan Infra Automation', 'Lynx Data Analytics']
AMBER-filtered: ['Quasar Self-Service Analytics', 'Helios Compliance Program']


## Ask the ELT Agent backend

In [6]:
scoped = backend.ask_question("Why is Phoenix Platform Modernization at risk?", AS_OF)
print(scoped["final_answer"][:400])
print("...\n")
assert "Answer:" in scoped["final_answer"]

2026-09-05 13:18:02.991 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


2026-09-05 13:18:02.992 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


{"request_id": "streamlit-chat-2026-09-15-4286153516235837112", "timestamp": "2026-09-05T17:18:02.993902+00:00", "event": "graph_node_start", "node": "classify_request"}
{"request_id": "streamlit-chat-2026-09-15-4286153516235837112", "timestamp": "2026-09-05T17:18:02.994150+00:00", "event": "graph_node_end", "node": "classify_request", "latency_ms": 0.01}
{"request_id": "streamlit-chat-2026-09-15-4286153516235837112", "timestamp": "2026-09-05T17:18:02.994549+00:00", "event": "graph_node_start", "node": "fetch_delivery_data"}
{"request_id": "streamlit-chat-2026-09-15-4286153516235837112", "timestamp": "2026-09-05T17:18:02.997699+00:00", "event": "records_retrieved", "source": "Jira", "issue_count": 37, "sprint_count": 3, "partial_failure": false}
{"request_id": "streamlit-chat-2026-09-15-4286153516235837112", "timestamp": "2026-09-05T17:18:02.997756+00:00", "event": "graph_node_end", "node": "fetch_delivery_data", "latency_ms": 3.17}
{"request_id": "streamlit-chat-2026-09-15-42861535162

## Every page, run headlessly via `AppTest`

This executes each page's real `.py` file — the same code a browser session would run — in Streamlit's simulated runtime, and reports any exception raised during the run. `AppTest.from_file` resolves a relative path against the CALLER's own file location (here, a temp kernel file), not the working directory — so absolute paths are used explicitly below.

In [7]:
from streamlit.testing.v1 import AppTest

pages = [str(PROJECT_ROOT / "app/ELT_Intelligence_Agent.py")] + sorted(str(p) for p in (PROJECT_ROOT / "app/pages").glob("*.py"))
results = {}
for page in pages:
    at = AppTest.from_file(page)
    at.run(timeout=60)
    results[page] = list(at.exception)

for page, exceptions in results.items():
    status = "OK" if not exceptions else f"FAILED: {[e.value for e in exceptions]}"
    print(f"  {Path(page).name:32s} {status}")

assert all(not e for e in results.values()), "one or more pages raised an exception"

2026-09-05 13:18:03.087 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


{"request_id": "streamlit-portfolio-2026-09-05", "timestamp": "2026-09-05T17:18:03.755274+00:00", "event": "graph_node_start", "node": "classify_request"}
{"request_id": "streamlit-portfolio-2026-09-05", "timestamp": "2026-09-05T17:18:03.755830+00:00", "event": "graph_node_end", "node": "classify_request", "latency_ms": 0.01}
{"request_id": "streamlit-portfolio-2026-09-05", "timestamp": "2026-09-05T17:18:03.756371+00:00", "event": "graph_node_start", "node": "fetch_delivery_data"}
{"request_id": "streamlit-portfolio-2026-09-05", "timestamp": "2026-09-05T17:18:03.777158+00:00", "event": "records_retrieved", "source": "Jira", "issue_count": 270, "sprint_count": 20, "partial_failure": false}
{"request_id": "streamlit-portfolio-2026-09-05", "timestamp": "2026-09-05T17:18:03.777237+00:00", "event": "graph_node_end", "node": "fetch_delivery_data", "latency_ms": 20.82}
{"request_id": "streamlit-portfolio-2026-09-05", "timestamp": "2026-09-05T17:18:03.777710+00:00", "event": "graph_node_start"

2026-09-05 13:18:03.966 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


2026-09-05 13:18:04.355 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


2026-09-05 13:18:04.669 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


...

2026-09-05 13:18:05.402 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


2026-09-05 13:18:05.766 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


2026-09-05 13:18:06.661 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


2026-09-05 13:18:06.951 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


  ELT_Intelligence_Agent.py        OK
  1_📊_Executive_Overview.py        OK
  2_📁_Project_Portfolio.py         OK
  3_🏃_Sprint_Health.py             OK
  4_💰_Financial_Health.py          OK
  5_⚠️_Risk_and_Blockers.py        OK
  6_📈_Trends.py                    OK
  7_💬_Ask_the_ELT_Agent.py         OK
  8_🔍_Data_Quality_Audit.py        OK


## Real server: `streamlit run` + health check

A final layer beyond the simulated harness — an actual `uvicorn`-backed process, confirmed via its own health endpoint rather than assumed from a clean exit code.

In [8]:
import subprocess
import time
import urllib.request

proc = subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", "app/ELT_Intelligence_Agent.py", "--server.headless", "true", "--server.port", "8766"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, cwd=str(PROJECT_ROOT),
)

health = None
for _ in range(20):
    try:
        health = urllib.request.urlopen("http://localhost:8766/_stcore/health", timeout=1).read().decode()
        break
    except Exception:
        time.sleep(1)

print(f"Health check: {health!r}")
assert health == "ok"

proc.terminate()
proc.wait(timeout=10)
print("Server stopped.")

Health check: 'ok'


Server stopped.


## Validation checks

- [x] `run_portfolio_query`/`ask_question` return real pipeline results, cached correctly by their inputs
- [x] Every DataFrame helper produces the expected shape and never fabricates a value (missing assignee -> "NOT AVAILABLE", not blank)
- [x] `apply_project_filters` intersects multiple active filters rather than unioning them
- [x] Every one of the 9 app files (entry point + 8 pages) runs with zero exceptions under `AppTest`
- [x] A real `streamlit run` process starts, serves `/_stcore/health` as `"ok"`, and stops cleanly

## Testing

`tests/test_backend.py` (19 tests) covers the same DataFrame/filter functions as unit tests; this notebook additionally exercises the full page set via `AppTest`, which pytest doesn't need to duplicate (it's exploratory/integration-style, best read here).

## Next step

Phase 11: the accuracy validation framework (Section 13's ten checks), consolidated and tested end to end — `13_accuracy_validation.ipynb`.